# Week 12 Practice: Classes, Objects, and a Robot

Run each cell in order, then change things and re-run. The last section builds the class you'll extend in the Block 4 capstone.

## 1. Why bother?

Here is the same data two ways. Try sorting `titles` in the first version and watch the authors stop matching.

In [ ]:
# Parallel lists - fragile
titles  = ["Dune", "Beloved", "Neuromancer"]
authors = ["Herbert", "Morrison", "Gibson"]

print(titles[1], "by", authors[1])

In [ ]:
# One object per book - each book keeps its own data together
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

books = [Book("Dune", "Herbert"), Book("Beloved", "Morrison")]
print(books[1].title, "by", books[1].author)

## 2. `__init__` and `self`

`__init__` runs automatically when you create an object. `self` is the object being worked on.

In [ ]:
class Student:
    def __init__(self, name, program="Undeclared"):
        self.name = name
        self.program = program
        self.credits = 0        # every student starts here

ana = Student("Ana", "MLIS")
ben = Student("Ben")

print(ana.name, "-", ana.program, "-", ana.credits)
print(ben.name, "-", ben.program, "-", ben.credits)

## 3. Methods

A method is a function that belongs to the class and can see `self`.

In [ ]:
class Student:
    def __init__(self, name, program="Undeclared"):
        self.name = name
        self.program = program
        self.credits = 0

    def enroll(self, credit_hours):
        self.credits += credit_hours
        return self.credits

    def standing(self):
        if self.credits >= 30:
            return "graduating"
        elif self.credits >= 12:
            return "in progress"
        return "just starting"


ana = Student("Ana", "MLIS")
ana.enroll(12)
ana.enroll(9)
print(ana.credits, "-", ana.standing())

## 4. `__str__` makes objects printable

Without it you get `<__main__.Student object at 0x...>`. Add one to every class you write.

In [ ]:
class Student:
    def __init__(self, name, program):
        self.name = name
        self.program = program

    def __str__(self):
        return f"{self.name} ({self.program})"


print(Student("Ana", "MLIS"))

## 5. Inheritance

`DVD` and `ReferenceBook` reuse everything from `LibraryItem` and change only what differs.

In [ ]:
class LibraryItem:
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id
        self.checked_out = False

    def check_out(self):
        if self.checked_out:
            return f"{self.title} is already out"
        self.checked_out = True
        return f"Checked out: {self.title}"

    def loan_period(self):
        return 21

    def __str__(self):
        return f"[{self.item_id}] {self.title}"


class DVD(LibraryItem):
    def loan_period(self):
        return 7


class ReferenceBook(LibraryItem):
    def check_out(self):
        return f"{self.title} is reference - library use only"

In [ ]:
items = [
    LibraryItem("Dune", "B001"),
    DVD("Arrival", "D014"),
    ReferenceBook("Oxford English Dictionary", "R002"),
]

# The loop does not know or care which class each item is
for item in items:
    print(item)
    print("  ", item.check_out())
    print("  ", item.loan_period(), "day loan")

## 6. `super()` extends instead of replacing

In [ ]:
class Vehicle:
    def __init__(self, make, model):
        self.make = make
        self.model = model

    def describe(self):
        return f"{self.make} {self.model}"


class ElectricCar(Vehicle):
    def __init__(self, make, model, range_miles):
        super().__init__(make, model)     # parent setup first
        self.range_miles = range_miles

    def describe(self):
        return f"{super().describe()} - {self.range_miles} mile range"


print(ElectricCar("Tesla", "Model 3", 272).describe())

## 7. Properties guard your data

The last line raises a `ValueError` on purpose. That is the point - the object refuses to enter an invalid state.

In [ ]:
class Thermostat:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32


t = Thermostat(20)
print(t.celsius, "C =", t.fahrenheit, "F")

t.celsius = 25
print(t.celsius, "C =", t.fahrenheit, "F")

In [ ]:
# Uncomment to see the guard work
# t.celsius = -300

## 8. Composition: objects holding objects

A DVD **is a** library item (inheritance). A library **has** items (composition).

In [ ]:
class Library:
    def __init__(self, name):
        self.name = name
        self.items = []

    def add(self, item):
        self.items.append(item)

    def available(self):
        return [i for i in self.items if not i.checked_out]

    def __len__(self):
        return len(self.items)


crown = Library("Rebecca Crown Library")
crown.add(LibraryItem("Dune", "B001"))
crown.add(DVD("Arrival", "D014"))
crown.items[0].check_out()

print(f"{crown.name}: {len(crown)} items, {len(crown.available())} available")

## 9. Looking ahead to Block 4

This is the seed of your capstone robot. Run it, then try adding `turn_left()`.

In [ ]:
class Robot:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y
        self.direction = "north"
        self.log = []

    def move_forward(self):
        if self.direction == "north":
            self.y += 1
        elif self.direction == "south":
            self.y -= 1
        elif self.direction == "east":
            self.x += 1
        else:
            self.x -= 1
        self.log.append(f"moved {self.direction} to ({self.x}, {self.y})")

    def turn_right(self):
        order = ["north", "east", "south", "west"]
        self.direction = order[(order.index(self.direction) + 1) % 4]
        self.log.append(f"turned right, now facing {self.direction}")

    def __str__(self):
        return f"Robot at ({self.x}, {self.y}) facing {self.direction}"


robot = Robot()
robot.move_forward()
robot.turn_right()
robot.move_forward()
robot.move_forward()

print(robot)
for entry in robot.log:
    print(" ", entry)

---

## Your Turn

**Exercise 1.** Add a `turn_left()` method to `Robot`. Test that four left turns return it to facing north.

**Exercise 2.** Write a `Playlist` class with `add(song)`, `remove(song)`, a `__len__`, and a `__str__` that reads like `"Study Mix (3 songs)"`.

**Exercise 3.** Write a `BankAccount` class where `withdraw()` refuses to overdraw and every transaction is appended to `self.history`. Print a statement at the end.

**Exercise 4.** Start from `LibraryItem` and add a new subclass `Periodical` with a 3-day loan period and an `issue_date` attribute. Add one to the `items` list above and re-run the loop - it should work without changing the loop at all.

**Exercise 5.** Give `Robot` a `battery` attribute that starts at 100 and drops by 5 on every move. Make `move_forward()` refuse to move when the battery hits zero.

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4

In [ ]:
# Exercise 5